# 04 · Cross-architecture results — figures and tables, YOLO26-seg vs. U-Net vs. SAM 3 (ISIC 2018 Task 1)

> **For the manuscript use `~/projects/analysis_outputs/notebooks/04_master_cross_model_comparison.ipynb`** (the paper
> workspace); this repository notebook remains the per-repository cross-architecture view.

Master analysis notebook for the thesis. Apart from section 0 (Phase 1 masks side by side, available as soon as the
masks are exported), it reads **only** the Phase 5 outputs of the three 5-phase pipelines and produces the
cross-architecture tables (LaTeX + CSV), the paired statistics and the figures. **No training, no GPU, no
re-evaluation** — run it after `run_pipeline*.sh --phases 5` has finished in the three repositories and the
`logs/<pipeline>/` folders are on this machine.

| Output | Content |
|---|---|
| Table 1 | Test accuracy (DSC, JSI, ISIC JSI$_{0.65}$, Boundary IoU, NSD, HD95) with bootstrap 95 % CI |
| Table 2 | Batch-1 efficiency: params, GFLOPs, median / P95 latency, FPS, peak VRAM, real-time criterion |
| Table 3 | Effect of HPO (Optimized − Baseline, paired) for every model |
| Table 4 | FP16 vs. FP32: accuracy cost and speed-up |
| Table 5 | Training cost per phase (GPU hours) |
| Table 6 | Phase 2 cross-validation vs. test (generalisation check) |
| Table 7 | Paired cross-architecture tests on per-image DSC (all pairs, Holm-adjusted) |
| Figure 1 | Accuracy (DSC, JSI) vs. latency, FPS and parameters, with the Pareto front |
| Figure 2 | Training time and inference time (median → P95) with the real-time budget |
| Figure 3 | Per-image score distributions |
| Figure 4 | Pairwise ΔDSC matrix with Holm-adjusted significance |
| Figure 5 | Boundary metrics (Boundary IoU, NSD, HD95) with 95 % CI |
| Figure 6 | Peak VRAM (model allocator vs. whole process incl. CUDA context) |

**Where things are.** This notebook and `results_aggregator.py` live in the folder that contains the three
repositories (`sandbox_yolo26/`, `sandbox_unet/`, `sandbox_sam3/`; any letter case), or in `<repo>/analysis/`. The
pipelines are found automatically under `<repo>/logs/<PIPELINE_NAME>/`; override with `$YOLO26_PIPELINE_DIR`,
`$UNET_PIPELINE_DIR`, `$SAM3_PIPELINE_DIR`. Everything is written to `<root>/analysis_outputs/exports/repo_notebook_04/`
(`tables/*.tex|csv`, `figures/*.pdf|png`, `data/*.csv`). The same outputs can be produced headless with
`python results_aggregator.py`.

**LaTeX.** Tables use `booktabs` and `\resizebox` → `\usepackage{booktabs,graphicx}`; include with `\input{...}`.

**Requirements.** `numpy`, `pandas`, `scipy`, `matplotlib` (no deep-learning stack).

In [ ]:
# ---- Configuration ----------------------------------------------------------
PIPELINE_NAME = "pipeline_final_v1"   # logs/<PIPELINE_NAME>/ in every repository
DEPLOY_VARIANT = "optimized"          # variant compared across architectures ("optimized" = Phase 4, chosen a priori)
PRECISION = "fp32"                    # primary precision (training was FP32); FP16 is in Table 4
REALTIME_FPS = 30                     # real-time criterion: end-to-end P95 latency <= 1000 / REALTIME_FPS ms
N_BOOT = 2000                         # bootstrap resamples (paired CIs), seeded
ROOT = None                           # folder holding the three repositories; None = auto-discover

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import results_aggregator as agg  # noqa: E402  (next to this notebook)

pd.set_option("display.max_columns", 40, "display.width", 200)
agg.setup_style()
PHASE5_MISSING = "⚠️ Phase 5 results not yet generated. Training is likely still in progress. Skipping cell execution."
try:
    art = agg.build(ROOT, PIPELINE_NAME, DEPLOY_VARIANT, PRECISION, REALTIME_FPS, n_boot=N_BOOT)
except FileNotFoundError as err:   # no pipeline has written its Phase 5 summary yet
    art, missing_reason = None, str(err)
PHASE5_READY = art is not None
if not PHASE5_READY:
    print(PHASE5_MISSING)
    print("[info]", missing_reason)
else:
    written = agg.write_tables(art)
    print("root     :", art.root)
    for k, p in art.pipes.items():
        print(f"pipeline : {agg.ARCH_BY_KEY[k].label:<11} {p.root}")
    print("outputs  :", art.out)
    print(f"systems  : {list(art.sel.system)}  ({DEPLOY_VARIANT}, {PRECISION})")
    print(f"tables   : {len(written)} written to {art.out / 'tables'}")
    for w in art.warnings:
        print("WARNING  :", w)

## 0. Phase 1 side by side — U-Net vs. YOLO26-nano vs. SAM 3 (official validation split)

Original image, ground truth and the Phase 1 (Baseline) mask of each model on the same validation images: 3 random
images and the 3 with the largest JSI spread between the models. Each model's mask is produced by its own pipeline
rule (U-Net: probability map upsampled, threshold 0.5; YOLO26 and SAM 3: top-1 instance, `PIXEL_CONF`) and scored
with the shared `segmentation_metrics`. **Independent of Phase 5** — it needs only the exported masks:

```bash
cd ~/projects    # CPU only (no --gpus): never competes with a training run
docker run --rm --cpus 8 --user $(id -u):$(id -g) -e HOME=/tmp -v $PWD/sandbox_unet/unet:/workspace/unet:ro \
  -v $PWD/sandbox_unet/datasets:/workspace/datasets:ro -v $PWD/sandbox_unet/logs:/workspace/logs \
  -v $PWD/sandbox_yolo26/datasets/isic2018_task1_official:/workspace/yolo26_dataset:ro \
  -w /workspace/unet --entrypoint python unet_ft export_phase1_val_masks.py
docker run --rm --cpus 8 --user $(id -u):$(id -g) -e HOME=/tmp -e YOLO_CONFIG_DIR=/tmp \
  -v $PWD/sandbox_yolo26/yolo26_seg:/workspace/yolo26_seg:ro -v $PWD/sandbox_yolo26/datasets:/workspace/datasets:ro \
  -v $PWD/sandbox_yolo26/logs:/workspace/logs -w /workspace/yolo26_seg --entrypoint python yolo26_ft \
  export_phase1_val_masks.py --models nano
docker run --rm --cpus 8 --user $(id -u):$(id -g) -e HOME=/tmp -v $PWD/sandbox_sam3/sam3_seg:/workspace/sam3_seg:ro \
  -v $PWD/sandbox_sam3/sam3:/workspace/sam3:ro -v $PWD/sandbox_sam3/datasets:/workspace/datasets:ro \
  -v $PWD/sandbox_sam3/logs:/workspace/logs -w /workspace/sam3_seg -e PYTHONPATH=/workspace \
  --entrypoint python sam3_ft export_phase1_val_masks.py --source dump
```

SAM 3 `--source dump` (CPU) uses the predictions of its **last** validation epoch, labelled with that epoch in the
figure; once a GPU is free, `--source best` (add `--gpus device=N`) exports the selected checkpoint instead.
Saved as `<folder holding the repositories>/analysis_outputs/exports/repo_notebook_04/figures/fig0_phase1_side_by_side.pdf|png`.

In [ ]:
# Independent of Phase 5: reads <repo>/logs/<PIPELINE_NAME>/phase1_val_masks/ of each repository
phase1_root = agg.discover_root(ROOT)
fig = agg.fig_phase1_side_by_side(phase1_root, PIPELINE_NAME, n_random=3, n_disagree=3,
                                  save=agg.save_to(phase1_root / "analysis_outputs" / "exports" / "repo_notebook_04" / "figures"))
if fig is None:
    print("⚠️ Phase 1 validation masks not yet exported. Skipping the side-by-side figure "
          "(run export_phase1_val_masks.py in each repository, commands above).")
else:
    plt.show()
    for s in agg.phase1_masks(phase1_root, PIPELINE_NAME):
        print(f"{s['label']:<22} {s['meta'].get('source', '')}")
    print("saved:", phase1_root / "analysis_outputs" / "exports" / "repo_notebook_04" / "figures" / "fig0_phase1_side_by_side.pdf|png")

## Pipeline notes to disclose

Protocol deviations recorded by the pipelines themselves (`protocol_notes` and `warnings` of every
`summary/final_results.json`). Everything printed here belongs in the Methods or Limitations section.

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    for k, p in art.pipes.items():
        for n in p.report.get("protocol_notes", []):
            print(f"[{agg.ARCH_BY_KEY[k].label}] NOTE: {n}")
    contended = [s for s, c in zip(art.sel.system, art.sel.get("contended", [None] * len(art.sel))) if pd.notna(c) and bool(c)]
    print("contended benchmarks (re-run Phase 5b on an idle GPU before quoting):", contended or "none")

## Table 1 — Test-set accuracy

Per-image mean over the 1,000 official test images with the seeded bootstrap 95 % CI written by each pipeline's
Phase 5a (identical metric code in the three repositories: `segmentation_metrics.py`). Best value per column in
bold in the LaTeX file. HD95 is in pixels at the dataset resolution (longer side ≤ 1,024 px); a missed lesion scores
the image diagonal (Metrics Reloaded: penalise, never drop).

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    display(art.tables["accuracy"])

## Table 2 — Efficiency at batch = 1 and the real-time criterion

Forward = network only (CUDA events); end-to-end = decoded image in host memory → binary mask at dataset resolution
in host memory (the mask that is scored), P95 over 100 distinct test images. VRAM: PyTorch allocator peak (the
model) and the driver-level process peak (CUDA context and allocator cache included — what a device must provide).
GFLOPs at each model's native input: YOLO26 640 px, U-Net 256 px, SAM 3 1008 px (thop for the CNNs,
`FlopCounterMode` for SAM 3, which counts the attention products thop cannot see).

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    display(art.tables["efficiency"])

## Figure 1 — Accuracy vs. efficiency

The central figure of the "lightweight vs. foundation model" argument: test DSC and JSI (95 % CI) against the measured
latency, throughput and parameter count. One colour per architecture; YOLO26 sizes are joined n → x; the dotted line
is the Pareto front (no other model is both faster/smaller and more accurate).

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    fig = agg.fig_accuracy_vs_efficiency(art.sel, metrics=("dsc", "jsi"), save=art.save_fig)
    plt.show()
    display(art.tables["accuracy_numeric"].merge(art.tables["efficiency_numeric"], on=["system", "arch_label"]).round(4))

## Figure 2 — Training and inference time

(a) GPU hours of the whole 5-phase protocol (Phases 1–4) and of the final model alone (Phase 4). (b) Median batch-1
latency with a line to its P95, forward vs. end-to-end and FP32 vs. FP16, against the real-time budget.

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    fig = agg.fig_training_inference_time(art.sel, art.tables["training"], REALTIME_FPS, save=art.save_fig)
    plt.show()
    display(art.tables["training"].round(2))

## Figure 3 — Per-image distributions

Box = IQR, centre line = median, whiskers = P5–P95, ◆ = mean. The mean (the reported figure) and the tails
(failures) can tell different stories — read with Table 1's "Missed" column.

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    fig = agg.fig_per_image_distributions(art.sel, art.per_image, save=art.save_fig)
    plt.show()

## Paired statistics across architectures

All systems are scored on the **same** 1,000 test images, so comparisons are paired by ISIC ID. Omnibus: Friedman
test per metric (Kendall's W = effect size; mean rank 1 = best). Pairs: mean difference with a seeded paired
bootstrap 95 % CI, two-sided Wilcoxon signed-rank test, matched-pairs rank-biserial r, and Holm-adjusted p-values
within each metric (all pairs = one family). Report the CI of the difference, not only the p-value: with
n = 1,000, clinically negligible differences can be "significant".

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    display(art.friedman.round(4))

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
elif art.tests.empty:
    print("⚠️ Paired cross-architecture tests need Phase 5 results from at least two architectures. Skipping cell execution.")
else:
    display(agg.pairwise_table(art.tests, "dsc"))

## Figure 4 — Pairwise ΔDSC matrix

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
elif art.tests.empty:
    print("⚠️ Paired cross-architecture tests need Phase 5 results from at least two architectures. Skipping cell execution.")
else:
    systems = [s for s in art.sel.system if s in art.per_image]
    fig = agg.fig_pairwise_matrix(art.tests, systems, "dsc", save=art.save_fig)
    plt.show()
    for m in ("jsi", "biou", "hd95"):   # the other metrics: same layout, saved for the supplementary material
        plt.close(agg.fig_pairwise_matrix(art.tests, systems, m, save=art.save_fig))

## Figure 5 — Boundary metrics (Metrics Reloaded)

Boundary IoU (band = 2 % of the image diagonal), NSD (tolerance = 1 % of the diagonal) and HD95, per-image mean with
bootstrap 95 % CI. Overlap scores are dominated by the lesion interior; these measure contour adherence.

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
elif art.tests.empty:
    print("⚠️ Paired cross-architecture tests need Phase 5 results from at least two architectures. Skipping cell execution.")
else:
    fig = agg.fig_boundary_metrics(art.sel, save=art.save_fig)
    plt.show()
    display(art.tests[art.tests.metric.isin(["biou", "nsd", "hd95"])][["metric", "A", "B", "delta", "ci95_low", "ci95_high", "p_holm"]].round(4))

## Figure 6 — Peak GPU memory

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    fig = agg.fig_memory(art.sel, save=art.save_fig)
    plt.show()

## Table 3 — Effect of HPO (Optimized − Baseline)

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    display(art.tables["hpo_gain"])

## Table 4 — FP16 vs. FP32

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    display(art.tables["precision"].round(4))

## Table 6 — Cross-validation vs. test

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    display(art.tables["cv_vs_test"])

## Key numbers for the Results section

Generated from the tables above — copy, then rephrase. Every number is traceable to `analysis_outputs/exports/repo_notebook_04/`.

In [ ]:
if not PHASE5_READY:
    print(PHASE5_MISSING)
else:
    for line in agg.key_numbers(art):
        print("•", line)
    print("\nFiles:")
    for f in sorted((art.out / "tables").glob("*.tex")) + sorted((art.out / "figures").glob("*.pdf")):
        print("  ", f.relative_to(art.out))

## Reading notes (keep these caveats in the thesis)

* **Resolution.** Every model is scored at the dataset resolution against the same official masks, but each runs at
  its own native input (U-Net 256, YOLO26 640, SAM 3 1008 px). Part of the accuracy and cost differences is the input
  resolution — state it.
* **GFLOPs** are reported at the native input; the U-Net's GFLOPs at 640 px are in `data/systems_selected.csv`
  (`gflops_640`) for a resolution-matched comparison. thop (CNNs) and `FlopCounterMode` (SAM 3) agree on convolutions
  and linear layers; only the latter counts attention products.
* **Budgets.** YOLO26 and U-Net: 120 epochs, HPO 30 × 30; SAM 3: 30 epochs, HPO 10 × 10 (compute). None uses early
  stopping. See `methodology_notes.md`.
* **Latency** depends on the GPU, driver and software stack (recorded in each efficiency JSON); quote the GPU model
  with every number, and never the per-image `infer_ms` column of Phase 5a (a by-product, not a benchmark).
* **Variant.** The comparison uses the variant fixed in advance (`DEPLOY_VARIANT`); Table 3 shows whether HPO helped.